> **Note:** This notebook defines an Airflow DAG for the sandbox environment.
> The code is meant to be read and understood — full execution requires Airflow's scheduler and infrastructure.
> Use the [sandbox Jupyter](https://sandbox.learnmlops.ops4life.com) to run DAGs end-to-end.


# Dataset Pipeline: Orchestrating with an Airflow DAG

This notebook mirrors the **Dataset Pipeline** guide at [learnmlops.ops4life.com/guides/dataset-pipeline/](https://learnmlops.ops4life.com/guides/dataset-pipeline/).

Write a self-contained Airflow DAG that runs the full employee attrition pipeline — ingest → validate → feature engineering → train — as scheduled tasks. Deploy the DAG to the shared workspace so Airflow picks it up automatically.

**What we'll cover:**
1. Understand the pipeline DAG structure
2. Write the DAG definition
3. Deploy to `/workspace/dags/`
4. Validate syntax
5. Trigger a run via the Airflow UI

In [ ]:
import os
import textwrap

# Create the DAGs directory on the shared Docker volume.
# Airflow's scheduler container mounts the same /workspace/dags path,
# so any .py file written here is automatically discovered within 30 seconds.
os.makedirs('/workspace/dags', exist_ok=True)

# Verify Airflow is reachable before proceeding.
# 'airflow version' is a lightweight CLI check — it fails immediately if
# Airflow isn't installed or the PATH isn't configured correctly.
import subprocess
result = subprocess.run(['airflow', 'version'], capture_output=True, text=True)
print(result.stdout.strip())


## Step 1: Understand the pipeline DAG structure

The pipeline has four sequential stages:

```
ingest_data → validate_data → feature_engineer → train_model
```

Each stage is a `PythonOperator` task. Airflow runs them in order, retrying on failure and recording each run's status in its metadata database.

In [ ]:
dag_code = textwrap.dedent('''
    from datetime import datetime, timedelta
    from airflow import DAG
    from airflow.operators.python import PythonOperator
    import pandas as pd
    import numpy as np
    import os

    DATA_DIR = "/workspace/pipeline-data"
    os.makedirs(DATA_DIR, exist_ok=True)

    default_args = {
        "owner": "mlops",
        "retries": 1,
        "retry_delay": timedelta(minutes=2),
    }


    def ingest_data(**context):
        np.random.seed(42)
        n = 1470
        df = pd.DataFrame({
            "Age":                      np.random.randint(22, 60, n),
            "Attrition":                np.random.choice(["Yes", "No"], n, p=[0.16, 0.84]),
            "BusinessTravel":           np.random.choice(["Non-Travel", "Travel_Rarely", "Travel_Frequently"], n),
            "DailyRate":                np.random.randint(100, 1500, n),
            "Department":               np.random.choice(["Sales", "Research & Development", "Human Resources"], n),
            "DistanceFromHome":         np.random.randint(1, 30, n),
            "Education":                np.random.randint(1, 5, n),
            "EmployeeNumber":           np.arange(1, n + 1),
            "EnvironmentSatisfaction":  np.random.randint(1, 4, n),
            "Gender":                   np.random.choice(["Male", "Female"], n),
            "HourlyRate":               np.random.randint(30, 100, n),
            "JobInvolvement":           np.random.randint(1, 4, n),
            "JobLevel":                 np.random.randint(1, 5, n),
            "JobRole":                  np.random.choice(["Sales Executive", "Research Scientist", "Laboratory Technician", "Manager"], n),
            "JobSatisfaction":          np.random.randint(1, 4, n),
            "MaritalStatus":            np.random.choice(["Single", "Married", "Divorced"], n),
            "MonthlyIncome":            np.random.randint(1500, 20000, n),
            "OverTime":                 np.random.choice(["Yes", "No"], n),
            "PerformanceRating":        np.random.randint(3, 5, n),
            "RelationshipSatisfaction": np.random.randint(1, 4, n),
            "TotalWorkingYears":        np.random.randint(0, 30, n),
            "WorkLifeBalance":          np.random.randint(1, 4, n),
            "YearsAtCompany":           np.random.randint(0, 20, n),
            "YearsInCurrentRole":       np.random.randint(0, 15, n),
            "YearsSinceLastPromotion":  np.random.randint(0, 10, n),
            "YearsWithCurrManager":     np.random.randint(0, 15, n),
        })
        path = f"{DATA_DIR}/raw_ingested.csv"
        df.to_csv(path, index=False)
        print(f"Ingested {len(df)} rows → {path}")


    def validate_data(**context):
        import pandera.pandas as pa
        from pandera import Column, Check

        df = pd.read_csv(f"{DATA_DIR}/raw_ingested.csv")

        schema = pa.DataFrameSchema({
            "Age":        Column(int, Check.between(22, 59)),
            "Attrition":  Column(str, Check.isin(["Yes", "No"])),
            "MonthlyIncome": Column(int, Check.gt(0)),
        }, strict=False, coerce=True)

        validated = schema(df, lazy=True)
        validated.to_csv(f"{DATA_DIR}/validated.csv", index=False)
        print(f"Validated {len(validated)} rows")


    def feature_engineer(**context):
        df = pd.read_csv(f"{DATA_DIR}/validated.csv")
        df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})
        df["OverTime"] = df["OverTime"].map({"Yes": 1, "No": 0})
        df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})
        df["BusinessTravel"] = df["BusinessTravel"].map({"Non-Travel": 0, "Travel_Rarely": 1, "Travel_Frequently": 2})
        df["MaritalStatus"] = df["MaritalStatus"].map({"Single": 0, "Married": 1, "Divorced": 2})
        df["OverallSatisfaction"] = (
            (df["JobSatisfaction"] + df["EnvironmentSatisfaction"] +
             df["RelationshipSatisfaction"] + df["WorkLifeBalance"]) / 4
        ).round().astype(int)
        df = df.drop(columns=["JobSatisfaction", "EnvironmentSatisfaction",
                               "RelationshipSatisfaction", "WorkLifeBalance"])
        df["PromotionStagnation"] = (df["YearsSinceLastPromotion"] / (df["YearsAtCompany"] + 1)).round(3)
        df["CareerVelocity"] = (df["JobLevel"] / (df["TotalWorkingYears"] + 1)).round(3)
        drop_cols = [c for c in ["EmployeeNumber", "JobRole", "Department",
                                  "DistanceFromHome", "YearsWithCurrManager"] if c in df.columns]
        df = df.drop(columns=drop_cols)
        df.to_csv(f"{DATA_DIR}/featured.csv", index=False)
        print(f"Feature engineering done: {df.shape}")


    def train_model(**context):
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.metrics import roc_auc_score
        import mlflow
        import mlflow.sklearn

        mlflow.set_tracking_uri("http://mlflow:5000")
        mlflow.set_experiment("attrition-pipeline-dag")

        df = pd.read_csv(f"{DATA_DIR}/featured.csv")
        X = df.drop(columns=["Attrition"])
        y = df["Attrition"]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ])

        with mlflow.start_run(run_name="dag-run"):
            pipeline.fit(X, y)
            auc = roc_auc_score(y, pipeline.predict_proba(X)[:, 1])
            mlflow.log_metric("train_auc", auc)
            mlflow.sklearn.log_model(pipeline, "model")
            print(f"Trained. Train AUC: {auc:.4f}")


    with DAG(
        dag_id="attrition_pipeline",
        default_args=default_args,
        schedule_interval="@daily",
        start_date=datetime(2025, 1, 1),
        catchup=False,
        tags=["mlops", "attrition"],
    ) as dag:
        t_ingest   = PythonOperator(task_id="ingest_data",       python_callable=ingest_data)
        t_validate = PythonOperator(task_id="validate_data",     python_callable=validate_data)
        t_features = PythonOperator(task_id="feature_engineer",  python_callable=feature_engineer)
        t_train    = PythonOperator(task_id="train_model",       python_callable=train_model)

        t_ingest >> t_validate >> t_features >> t_train
''').strip()

print("DAG code preview (first 30 lines):")
print("\n".join(dag_code.split("\n")[:30]))
print("...")

## Step 2: Understand key DAG parameters

- **`schedule_interval="@daily"`** — Airflow runs this DAG once per day
- **`catchup=False`** — skips historical runs if the DAG is enabled late
- **`start_date`** — the earliest date Airflow will schedule runs from
- **`default_args.retries=1`** — each task retries once on failure before the DAG run fails
- **`tags`** — used to filter DAGs in the Airflow UI

## Step 3: Deploy to the DAGs folder

In [ ]:
dag_path = '/workspace/dags/attrition_pipeline.py'

# Write the DAG Python file to the shared DAGs directory.
# Airflow parses this file directly — the dag_code string must be valid Python
# with a top-level DAG object. Airflow scans for DAG objects at import time.
with open(dag_path, 'w') as f:
    f.write(dag_code)

print(f'DAG deployed: {dag_path}')


Airflow scans `AIRFLOW__CORE__DAGS_FOLDER` (set to `/workspace/dags`) every 30 seconds. The `sandbox-notebooks` Docker volume is shared between the Jupyter and Airflow containers — so writing to `/workspace/dags/` here makes the DAG immediately visible to Airflow.

## Step 4: Validate DAG syntax

In [ ]:
# Validate DAG syntax by running the file directly with Python.
# This catches syntax errors and import errors before Airflow tries to parse it.
# Airflow reports parsing failures in the UI as a red banner on the DAG — this
# step surfaces them earlier so you don't have to wait for the next scheduler scan.
# returncode=0 means no errors; non-zero means something failed (check stderr).
result = subprocess.run(['python', dag_path], capture_output=True, text=True)
if result.returncode == 0:
    print('Syntax OK — no errors')
else:
    print('Syntax error:')
    print(result.stderr)


## Step 5: Trigger a run via the Airflow UI

1. Open [airflow.learnmlops.ops4life.com](https://airflow.learnmlops.ops4life.com)
2. Log in with username `admin` and the password from `/airflow/standalone_admin_password.txt`
3. Find **attrition_pipeline** in the DAG list (may take up to 30 seconds to appear)
4. Toggle the DAG on and click **Trigger DAG**
5. Open the **Grid** or **Graph** view to watch each task run in sequence

## Next Steps

- Continue to feature engineering: `02-pipeline/data-preparation/feature_engineering.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/dataset-pipeline/](https://learnmlops.ops4life.com/guides/dataset-pipeline/)